# Introduction

This project investigates gene expression differences between glioblastoma tumor samples and normal tissue samples from the TCGA-GBM dataset using Python and PyDESeq2.

This first notebook prepares the inputs for downstream analysis. STAR count files downloaded from the Genomic Data Commons (GDC) are combined into a count matrix, with genes as rows and samples as columns. A corresponding metadata table records the condition assigned to each sample.

Ensembl gene identifiers are retained for differential expression analysis, while a separate annotation table links these identifiers to gene symbols for interpreting and visualizing the results.

# Objectives

- Import gene-level STAR count files for tumor and normal samples.
- Extract raw, unstranded counts from each file.
- Standardize Ensembl identifiers by removing version suffixes.
- Combine sample counts into a single gene-by-sample matrix.
- Create sample metadata containing the condition labels.
- Prepare an Ensembl-to-gene-symbol annotation table.
- Export the inputs for quality control and differential expression analysis.

# Input 

- GDC STAR count files organized into Tumor and Normal folders.

# Outputs

| File | Contents |
|---|---|
| `count_matrix_ensembl.csv` | Raw gene counts, with Ensembl IDs as rows and samples as columns |
| `metadata.csv` | Sample identifiers and tumor/normal condition labels |
| `gene_annotation_GENCODE.csv` | Ensembl identifiers and corresponding gene symbols |

# Analysis Sequence

Count matrix construction → Quality control and filtering → Differential expression analysis

- This notebook prepares raw counts. Gene filtering and sample assessment are performed in Notebook 2, followed by normalization and differential expression analysis in Notebook 3.


In [3]:
## 1. Import Libraries and Define Project Paths
## 2. Locate Tumor and Normal STAR Count Files
## 3. Extract Raw Counts and Standardize Ensembl Identifiers
## 4. Construct the Gene-by-Sample Count Matrix
## 5. Create and Validate Sample Metadata
## 6. Build the Gene Annotation Table
## 7. Export Count Matrix, Metadata, and Annotation
## 8. Summary and Next Steps

In [4]:
## 1. Import Libraries and Define Project Paths

from pathlib import Path
import pandas as pd
import numpy as np
import os
import glob



To get and copy the path of a folder:

Method 1: Drag and Drop Open Finder. Find the folder you want. Open Terminal. Drag the folder from Finder into the Terminal window.

Method 2: Copy Pathname In Finder, right-click the folder. Hold the Option (⌥) key. You'll see "Copy 'FolderName' as Pathname"

In [8]:
## 2. Import Tumor and Normal STAR Count Files


GBM_PATH =  "/Users/olahassan/Desktop/Python/TCGA_GBM"

TUMOR_PATH = os.path.join(GBM_PATH, "Tumor")

NORMAL_PATH = os.path.join(GBM_PATH, "Normal")

OUTPUT_PATH = "/Users/olahassan/Desktop/Python/TCGA_GBM/results"

# add a apath to the project  

GBM_PROJECT = Path("/Users/olahassan/Desktop/Python/TCGA_GBM")

# validate all needed files exist

print(os.listdir(GBM_PATH))

['.DS_Store', 'data_processed', 'Tumor', 'results', 'TCGA-GBM-RNAseq-Differential-Expression-Analysis-using-Python', 'Normal', '.git', 'notebooks']


In [7]:
## 3. Extract Raw Counts and Standardize Ensembl Identifiers


## Convert tumor files into a vector

tumor_files = glob.glob(
    os.path.join(TUMOR_PATH, "*", "*.tsv"),
recursive=True
)
print(len(tumor_files))



## Convert normal files into a vector

normal_files = glob.glob(
    os.path.join(NORMAL_PATH, "**", "*.tsv"),
    recursive=True
)
print(len(normal_files))

#Create a vector for ll files
all_files = tumor_files + normal_files

len(all_files)


5
5


10

In [9]:
from pathlib import Path

all_files = [
    Path(f)
    for f in all_files
]

type(all_files[0])

pathlib.PosixPath

In [13]:


count_matrix = None
sample_info = []




In [14]:
## 4. Construct the Sample Count Matrix

#Create count matrix, 
#Use gene_id
#Remove Ensembl version
#Collapse duplicates BEFORE merging
#Keep only raw counts



for file in all_files:

    sample_id = file.parent.name

    condition = file.parent.parent.name

    print(condition, sample_id)

    df = pd.read_csv(
        file,
        sep="\t",
        comment="#"
    )

    df["gene_id"] = (
        df["gene_id"]
        .str.split(".")
        .str[0]
    )

    df = df[
        [
            "gene_id",
            "unstranded"
        ]
    ]

    # collapse duplicate gene IDs
    df = (
        df
        .groupby("gene_id")["unstranded"]
        .sum()
        .to_frame()
    )

    df.columns = [sample_id]


    if count_matrix is None:
        count_matrix = df
    else:
        count_matrix = count_matrix.join(
            df,
            how="inner"
        )


    sample_info.append(
        {
            "sample": sample_id,
            "condition": condition
        }
    )

Tumor 77c261f0-384c-4906-a57e-7c3b35c17041
Tumor 99ff2016-a97a-4142-b70a-1bb59d6264e9
Tumor d4649bc7-09d2-4d90-bafd-0bec72429b00
Tumor afbd1975-263b-404b-a738-886bf25c0223
Tumor 444a2332-9e52-4cf8-94d9-07a8b829633e
Normal 64a2ddbe-3614-409a-af16-170a1471db2a
Normal 4bed9101-07f4-4d76-b79f-6eb8de04bd19
Normal ddaed459-ab19-4f75-a1a9-72761287c6f5
Normal 477a4ae1-84c0-49f2-b3b3-4015b8c26f18
Normal c410d37e-e445-4f5f-8937-e6796dd52364


In [15]:
count_matrix.shape()

TypeError: 'tuple' object is not callable

In [16]:
count_matrix.head()

,77c261f0-384c-4906-a57e-7c3b35c17041,99ff2016-a97a-4142-b70a-1bb59d6264e9,d4649bc7-09d2-4d90-bafd-0bec72429b00,afbd1975-263b-404b-a738-886bf25c0223,444a2332-9e52-4cf8-94d9-07a8b829633e,64a2ddbe-3614-409a-af16-170a1471db2a,4bed9101-07f4-4d76-b79f-6eb8de04bd19,ddaed459-ab19-4f75-a1a9-72761287c6f5,477a4ae1-84c0-49f2-b3b3-4015b8c26f18,c410d37e-e445-4f5f-8937-e6796dd52364
gene_id,,,,,,,,,,
ENSG00000000003,2345,1550,2934,2420,2316,699,608,452,413,546
ENSG00000000005,9,3,10,2,7,5,7,5,0,9
ENSG00000000419,1843,1180,762,676,1059,971,986,928,795,1074
ENSG00000000457,606,748,950,697,1065,606,492,402,532,478
ENSG00000000460,340,434,442,364,783,81,81,61,146,101


In [17]:
#Check for duplicates

count_matrix.index.duplicated().sum()

np.int64(0)

In [44]:

#Create folder:
(GBM_PROJECT / "data_processed").mkdir( exist_ok=True)


In [93]:
#Save counts

count_matrix.to_csv(
    GBM_PROJECT / "data_processed" / "count_matrix_ensembl.csv"
)

In [18]:
## 5. Create and Validate Sample Metadata


metadata = pd.DataFrame(sample_info)

metadata = metadata.set_index("sample")

In [19]:
#check 
metadata.shape

(10, 1)

In [50]:
#save
metadata.to_csv(
    GBM_PROJECT / "data_processed" / "metadata.csv"
)

In [52]:
# Create another count matrix file with gene symbols instad of ensemble IDs, to use it for PyDESEq later

Tumor_file = glob.glob(
    os.path.join(TUMOR_PATH, "*", "*.tsv")
)[0]


Tumor_gene_map = pd.read_csv(
    Tumor_file,
    sep="\t",
    comment="#"
)[["gene_id","gene_name"]]

#Remove un needed columns 
Tumor_gene_map = Tumor_gene_map[
    Tumor_gene_map["gene_id"].str.startswith("ENSG")
]


#Remove Ensemble version numbers
Tumor_gene_map["ensembl_id"] = (
    Tumor_gene_map["gene_id"]
    .str.split(".")
    .str[0]
)

Tumor_gene_map.head()

,gene_id,gene_name,ensembl_id
4,ENSG00000000003.15,TSPAN6,ENSG00000000003
5,ENSG00000000005.6,TNMD,ENSG00000000005
6,ENSG00000000419.13,DPM1,ENSG00000000419
7,ENSG00000000457.14,SCYL3,ENSG00000000457
8,ENSG00000000460.17,C1orf112,ENSG00000000460


In [54]:
#Repeat for Normal

Normal_file = glob.glob(
    os.path.join(NORMAL_PATH, "*", "*.tsv")
)[0]


Normal_gene_map = pd.read_csv(
    Normal_file,
    sep="\t",
    comment="#"
)[["gene_id","gene_name"]]


Normal_gene_map = Normal_gene_map[
    Normal_gene_map["gene_id"].str.startswith("ENSG")
]


Normal_gene_map["ensembl_id"] = (
    Normal_gene_map["gene_id"]
    .str.split(".")
    .str[0]
)

Normal_gene_map.head()

,gene_id,gene_name,ensembl_id
4,ENSG00000000003.15,TSPAN6,ENSG00000000003
5,ENSG00000000005.6,TNMD,ENSG00000000005
6,ENSG00000000419.13,DPM1,ENSG00000000419
7,ENSG00000000457.14,SCYL3,ENSG00000000457
8,ENSG00000000460.17,C1orf112,ENSG00000000460


In [56]:
#Check Tumor and Normal mappings are identical (Very important)

Tumor_gene_map.shape, Normal_gene_map.shape

((60660, 3), (60660, 3))

In [58]:
(
Tumor_gene_map["ensembl_id"]
==
Normal_gene_map["ensembl_id"]
).all()

np.True_

In [70]:
## 6. Create gene annotation table

#using only one of the  files Tumor or Normal to create the final annotation table is correct,
#provided that Tumor and Normal files were generated from the same GDC workflow (STAR - Counts) and the same GENCODE reference version.
#since both are TCGA GDC files, they should both use the same annotation:

gene_annotation = Tumor_gene_map[
    ["ensembl_id","gene_name"]
].copy()


In [72]:
# rename

gene_annotation.columns = [
    "ensembl_id",
    "gene_symbol"
]

In [64]:
# Save the gene annotation to be used in PyDSEQ

gene_annotation.to_csv(GBM_PROJECT / "data_processed" / 
    "gene_annotation_GENCODE.csv",
    index=False
)

In [74]:
# Reset  count matrix 

#Remove older version of the count matrix
count_matrix.index = (
    count_matrix.index
    .str.split(".")
    .str[0]
)

#Then merge annotation and Set gene symbols as index

count_matrix_symbol = (
    count_matrix
    .reset_index()
    .merge(
        gene_annotation,
        left_on="gene_id",
        right_on="ensembl_id",
        how="left"
    )
)


In [76]:
count_matrix_symbol.head()

,gene_id,77c261f0-384c-4906-a57e-7c3b35c17041,99ff2016-a97a-4142-b70a-1bb59d6264e9,d4649bc7-09d2-4d90-bafd-0bec72429b00,afbd1975-263b-404b-a738-886bf25c0223,444a2332-9e52-4cf8-94d9-07a8b829633e,64a2ddbe-3614-409a-af16-170a1471db2a,4bed9101-07f4-4d76-b79f-6eb8de04bd19,ddaed459-ab19-4f75-a1a9-72761287c6f5,477a4ae1-84c0-49f2-b3b3-4015b8c26f18,c410d37e-e445-4f5f-8937-e6796dd52364,ensembl_id,gene_symbol
0,ENSG00000000003,2345,1550,2934,2420,2316,699,608,452,413,546,ENSG00000000003,TSPAN6
1,ENSG00000000005,9,3,10,2,7,5,7,5,0,9,ENSG00000000005,TNMD
2,ENSG00000000419,1843,1180,762,676,1059,971,986,928,795,1074,ENSG00000000419,DPM1
3,ENSG00000000457,606,748,950,697,1065,606,492,402,532,478,ENSG00000000457,SCYL3
4,ENSG00000000460,340,434,442,364,783,81,81,61,146,101,ENSG00000000460,C1orf112


In [78]:
# Handle duplicate gene symbols (This is important., DESeq2 needs unique rows.)


count_matrix_symbol = (
    count_matrix_symbol
    .groupby(count_matrix_symbol.index)
    .sum()
)

In [80]:
count_matrix_symbol.shape

(60664, 13)

In [82]:
count_matrix_symbol.head()

,gene_id,77c261f0-384c-4906-a57e-7c3b35c17041,99ff2016-a97a-4142-b70a-1bb59d6264e9,d4649bc7-09d2-4d90-bafd-0bec72429b00,afbd1975-263b-404b-a738-886bf25c0223,444a2332-9e52-4cf8-94d9-07a8b829633e,64a2ddbe-3614-409a-af16-170a1471db2a,4bed9101-07f4-4d76-b79f-6eb8de04bd19,ddaed459-ab19-4f75-a1a9-72761287c6f5,477a4ae1-84c0-49f2-b3b3-4015b8c26f18,c410d37e-e445-4f5f-8937-e6796dd52364,ensembl_id,gene_symbol
0,ENSG00000000003,2345,1550,2934,2420,2316,699,608,452,413,546,ENSG00000000003,TSPAN6
1,ENSG00000000005,9,3,10,2,7,5,7,5,0,9,ENSG00000000005,TNMD
2,ENSG00000000419,1843,1180,762,676,1059,971,986,928,795,1074,ENSG00000000419,DPM1
3,ENSG00000000457,606,748,950,697,1065,606,492,402,532,478,ENSG00000000457,SCYL3
4,ENSG00000000460,340,434,442,364,783,81,81,61,146,101,ENSG00000000460,C1orf112


In [152]:
#Save counts having the gene symbols and the ensembl Ids together

count_matrix_symbol.to_csv(
    GBM_PROJECT / "data_processed" / "count_matrix_ensembe_symbol.csv"
)

## 8. Summary and Next Steps

This notebook combined individual STAR count files into a gene-by-sample raw count matrix and prepared the corresponding sample metadata and gene annotation table. Ensembl identifiers were retained for downstream analysis, with gene symbols stored separately for interpretation and visualization.

The following files were exported:

- `count_matrix_ensembe_symbol.csv`
- `metadata.csv`
- `gene_annotation_GENCODE.csv`

### Next Steps

In **Notebook 2: Quality Control**, these inputs will be used to assess library sizes, filter low-count genes, and examine sample relationships through PCA and hierarchical clustering.

The resulting filtered raw count matrix will then be used for differential expression analysis with PyDESeq2 in **Notebook 3**.

## Workflow

**`count_matrix_ensembe_symbol.csv`**

↓  

**Notebook 2: Quality Control and Filtering**

↓  

**Notebook 3: Differential Expression with PyDESeq2**

↓  

**Merge DESeq2 Results with the Gene Annotation Table**

↓  

**Add Gene Symbols for Interpretation and Plotting**